# Données non séparables : score affine ou MLP

Ce notebook accompagne l'exercice 5.3. Un petit jeu de six données suffit pour faire apparaître deux limitations différentes : une perte ne détermine pas à elle seule la frontière, et un score affine ne peut pas représenter toutes les séparations.

Nous comparerons un SVM affine, un classificateur logistique affine et un MLP. La dernière comparaison utilise la même perte logistique pour isoler l'effet de la classe de fonctions.

## Parcours

1. [Un jeu non séparable par un score affine](#jeu-nonseparable)
2. [Deux critères affines](#criteres-affines)
3. [Un score non affine exact](#score-non-affine)
4. [Un MLP avec Flax NNX](#mlp-nonseparable)
5. [Comparer sans confondre perte et classe de fonctions](#comparaison-nonseparable)

In [1]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
import flax
from flax import nnx

print(f"JAX {jax.__version__}, Flax {flax.__version__}, Optax {optax.__version__}")

JAX 0.11.1, Flax 0.12.9, Optax 0.2.8


In [2]:
x_ns = jnp.array([
    [-2.0, -1.0],
    [-2.0,  1.0],
    [ 2.0,  0.0],
    [ 2.0, -1.0],
    [ 2.0,  1.0],
    [-2.0,  0.0],
])
z_ns = jnp.array([-1.0, -1.0, -1.0, 1.0, 1.0, 1.0])

def score_affine(parametres, x):
    return x @ parametres["w"] + parametres["b"]

def marges_affines(parametres, x, z):
    return z * score_affine(parametres, x)

<a id="jeu-nonseparable"></a>
## 1. Un jeu non séparable par un score affine

1. Représenter les données.
2. Observer les trois points d'abscisse $2$. Pourquoi l'affinité du score rend-elle leur classement exact impossible ?
3. Reprendre le même raisonnement pour les points d'abscisse $-2$.

In [ ]:
# À compléter.

<a id="criteres-affines"></a>
## 2. Deux critères affines

Ajuster le même score affine avec :

- la perte charnière régularisée ;
- la perte logistique régularisée.

Comparer la fréquence des erreurs, la valeur de chaque perte et les marges signées. Ces trois quantités ordonnent-elles nécessairement les solutions de la même manière ?

La fonction d'apprentissage est fournie afin que l'expérience reste centrée sur la comparaison des critères.

In [4]:
def critere_affine(parametres, x, z, lamb, nature):
    marges = marges_affines(parametres, x, z)
    if nature == 0:
        perte = jnp.mean(jax.nn.relu(1.0 - marges))
    else:
        perte = jnp.mean(jax.nn.softplus(-marges))
    return perte + 0.5 * lamb * jnp.sum(parametres["w"]**2)

def entrainer_affine(x, z, lamb, nature, *, iterations=3000, alpha=2.0e-2):
    parametres = {"w": jnp.zeros(x.shape[1]), "b": jnp.array(0.0)}
    transformation = optax.adam(alpha)
    etat = transformation.init(parametres)

    @jax.jit
    def pas(parametres, etat):
        fonction = lambda p: critere_affine(p, x, z, lamb, nature)
        valeur, gradient = jax.value_and_grad(fonction)(parametres)
        mises_a_jour, etat = transformation.update(gradient, etat, parametres)
        return optax.apply_updates(parametres, mises_a_jour), etat, valeur

    for _ in range(iterations):
        parametres, etat, valeur = pas(parametres, etat)
    return parametres, float(valeur)

def statistiques(scores, z):
    marges = z * scores
    return {
        "fréquence d'erreur": float(jnp.mean(marges <= 0.0)),
        "perte logistique": float(jnp.mean(jax.nn.softplus(-marges))),
        "perte charnière": float(jnp.mean(jax.nn.relu(1.0 - marges))),
        "marge minimale": float(jnp.min(marges)),
    }

In [ ]:
# À compléter.

<a id="score-non-affine"></a>
## 3. Un score non affine exact

Vérifier que
$$
F(x_1,x_2)=x_1(2x_2^2-1)
$$
classe correctement les six données. Représenter sa frontière de décision et expliquer pourquoi elle ne peut pas être celle d'un classificateur affine.

In [ ]:
# À compléter.

In [ ]:
# À compléter.

<a id="mlp-nonseparable"></a>
## 4. Un MLP avec Flax NNX

Construire un MLP scalaire à deux couches cachées et l'ajuster avec la perte logistique. Nous pénalisons la norme euclidienne de tous ses paramètres :
$$
\mathcal L(p)=\frac1{n_{\mathcal D}}\sum_i
\log(1+e^{-z_i\Phi(x_i,p)})+\frac\lambda2\lVert p\rVert^2.
$$

Comparer plusieurs initialisations si la solution trouvée ne classe pas immédiatement les six données.

In [ ]:
# À compléter.

<a id="comparaison-nonseparable"></a>
## 5. Comparer sans confondre perte et classe de fonctions

Représenter sur une même figure :

- le SVM affine ;
- le score affine ajusté avec la perte logistique ;
- le MLP ajusté avec cette même perte logistique.

La comparaison entre les deux derniers modèles isole l'effet de la classe de fonctions. La comparaison entre les deux premiers isole l'effet de la perte.

In [ ]:
# À compléter.

In [ ]:
# À compléter.

## Bilan

Pour analyser une expérience de classification, il faut séparer au moins deux choix : la fonction objectif et la classe de machines. Utiliser la même perte pour le score affine et pour le MLP est le moyen le plus direct de rendre visible le second choix.